# 01.1 — Choosing services and models: lab

The README argued about model selection. This notebook makes you *measure* it.

You will query the model catalog, then run one identical task against three
deployments — `gpt-4o-mini` (SLM), `gpt-4o` (frontier LLM), and `o4-mini`
(reasoning) — and record cost, latency, and correctness for each. At the end you
write down a choice and defend it with the numbers.

**Cost:** a few cents. Nothing created here bills by the hour.

**Prerequisites:** unit 00 complete. If `gpt-4o` or `o4-mini` are not deployed,
the lab detects that and skips them rather than failing.

## 1. Connect

Two folders deep now, so `parents[1]` reaches `ai103-learning/scripts`.

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / "scripts"))
from ai103 import cfg, credential, chat_client, project_client, show_usage

print("project  :", cfg["AZURE_AI_PROJECT_ENDPOINT"])
print("region   :", cfg["AZURE_LOCATION"])

## 2. What can this resource actually run?

Two different questions, two different APIs:

- **What is deployed?** — data plane, `AIProjectClient.deployments`
- **What *could* be deployed here?** — control plane, ARM
  (`Microsoft.CognitiveServices/accounts/models`)

Confusing the two is a common source of "the catalog shows it but I cannot use
it" confusion. The catalog is global; availability is per region.

In [ ]:
project = project_client()

deployed = {}
print(f"{'deployment':<28} {'model':<26} {'version':<12} type")
print("-" * 82)
for d in project.deployments.list():
    name = d.name
    model = getattr(d, "model_name", "") or ""
    version = getattr(d, "model_version", "") or ""
    deployed[name] = model
    print(f"{name:<28} {model:<26} {version:<12} {getattr(d, 'type', '')}")

Now the control plane. `list_models` returns every model the *region* can host,
with the SKUs it supports. The `skus` list is the deployment-type decision from
the README, expressed as data: `GlobalStandard`, `Standard`, `DataZoneStandard`,
`ProvisionedManaged`, `GlobalBatch`.

In [ ]:
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient

arm = CognitiveServicesManagementClient(credential(), cfg["AZURE_SUBSCRIPTION_ID"])

catalog = []
for m in arm.models.list(location=cfg["AZURE_LOCATION"]):
    md = m.model
    if md is None:
        continue
    catalog.append(
        {
            "name": md.name,
            "version": md.version,
            "format": md.format,
            "skus": sorted({s.name for s in (md.skus or [])}),
        }
    )

print(f"{len(catalog)} model versions deployable in {cfg['AZURE_LOCATION']}\n")

interesting = ("gpt-4o", "gpt-4o-mini", "o4-mini", "text-embedding-3-small", "Phi-4")
seen = set()
for entry in catalog:
    if entry["name"] in interesting and entry["name"] not in seen:
        seen.add(entry["name"])
        print(f"{entry['name']:<26} {entry['format']:<12} {', '.join(entry['skus'])}")

> **Exam note.** A model offering only `ProvisionedManaged` in your region cannot
> be run pay-per-token there. That is a *region* fact, not a *model* fact —
> the same model may offer `GlobalStandard` in `swedencentral`. Region
> availability is the first thing to check when a deployment is refused.

## 3. Define one task and one rubric

Benchmarks published by vendors tell you which models to *shortlist*. They never
tell you which to pick, because your prompt, your data, and your tolerance for
error are not in the benchmark.

The task below is deliberately chosen to sit on the boundary: it needs a little
arithmetic and a little constraint-checking, so a weak model plausibly fails.
The answer is verifiable, so scoring is objective rather than vibes.

In [ ]:
TASK = """A support team handles 3 tiers of tickets.

Tier 1: 1,200 tickets/month, 6 minutes each.
Tier 2:   450 tickets/month, 25 minutes each.
Tier 3:    80 tickets/month, 95 minutes each.

An agent deflects 40% of Tier 1 and 15% of Tier 2. It cannot touch Tier 3.
An engineer works 130 productive minutes per day, 20 days per month.

How many engineers are needed after the agent is deployed?
Reply with ONLY an integer, rounded up. No units, no words."""

# Worked by hand:
#   Tier 1 remaining: 1200 * 0.60 * 6  = 4,320 min
#   Tier 2 remaining:  450 * 0.85 * 25 = 9,562.5 min
#   Tier 3:             80 * 95        = 7,600 min
#   total 21,482.5 min / (130 * 20 = 2,600 min per engineer) = 8.26 -> 9
EXPECTED = 9

print(TASK)
print("\nexpected answer:", EXPECTED)

## 4. The harness

Three things make this measurement honest, and all three are easy to get wrong:

1. **Discard a warm-up call.** The first request to a deployment pays TLS setup
   and routing.
2. **Report the median, not the mean.** Global Standard routes to whatever region
   has capacity, so one outlier can double a mean.
3. **Branch on reasoning models.** `o4-mini` rejects `temperature` and requires
   `max_completion_tokens` instead of `max_tokens`. Production code needs this
   same branch.

In [ ]:
import re, time, statistics

client = chat_client()

# USD per token. Rates change constantly — replace with today's pricing page values.
PRICING = {
    "gpt-4o-mini": (0.15 / 1_000_000, 0.60 / 1_000_000),
    "gpt-4o": (2.50 / 1_000_000, 10.00 / 1_000_000),
    "o4-mini": (1.10 / 1_000_000, 4.40 / 1_000_000),
}

REASONING_MODELS = {"o4-mini", "o3", "o3-mini", "o1"}


def call(deployment: str):
    """One call. Returns (seconds, response) or raises."""
    kwargs = {"model": deployment, "messages": [{"role": "user", "content": TASK}]}
    if deployment in REASONING_MODELS:
        kwargs["max_completion_tokens"] = 4000  # must cover hidden reasoning tokens
    else:
        kwargs["temperature"] = 0.0
        kwargs["max_tokens"] = 50
    start = time.perf_counter()
    resp = client.chat.completions.create(**kwargs)
    return time.perf_counter() - start, resp


def parse_int(text: str):
    m = re.search(r"-?\d+", (text or "").replace(",", ""))
    return int(m.group()) if m else None


print("harness ready")

## 5. Run the comparison

Five scored trials per model after one discarded warm-up. Models that are not
deployed are skipped with a note rather than crashing the notebook.

In [ ]:
TRIALS = 5
candidates = [cfg["MODEL_MINI"], cfg["MODEL_CHAT"], cfg.get("MODEL_REASONING", "o4-mini")]

results = {}

for dep in candidates:
    if dep not in deployed:
        print(f"skip {dep!r} — not deployed in this project")
        continue
    try:
        call(dep)  # warm-up, discarded
    except Exception as exc:  # noqa: BLE001 - we want the message, not the traceback
        print(f"skip {dep!r} — {type(exc).__name__}: {exc}")
        continue

    lat, cost, correct, reasoning = [], [], 0, []
    for _ in range(TRIALS):
        secs, resp = call(dep)
        u = resp.usage
        rate_in, rate_out = PRICING.get(deployed[dep], (0.0, 0.0))
        lat.append(secs)
        cost.append(u.prompt_tokens * rate_in + u.completion_tokens * rate_out)
        correct += int(parse_int(resp.choices[0].message.content) == EXPECTED)
        details = getattr(u, "completion_tokens_details", None)
        reasoning.append(getattr(details, "reasoning_tokens", 0) or 0)

    results[dep] = {
        "p50_latency_s": statistics.median(lat),
        "max_latency_s": max(lat),
        "cost_per_call": statistics.median(cost),
        "accuracy": correct / TRIALS,
        "reasoning_tokens": statistics.median(reasoning),
    }
    print(f"done {dep}")

In [ ]:
print(f"{'deployment':<18}{'acc':>6}{'p50 s':>9}{'max s':>9}{'$/call':>12}{'$/100k':>11}{'reason tok':>12}")
print("-" * 77)
for dep, r in results.items():
    print(
        f"{dep:<18}"
        f"{r['accuracy']:>6.0%}"
        f"{r['p50_latency_s']:>9.2f}"
        f"{r['max_latency_s']:>9.2f}"
        f"{r['cost_per_call']:>12.6f}"
        f"{r['cost_per_call'] * 100_000:>11.2f}"
        f"{int(r['reasoning_tokens']):>12}"
    )

### Reading the table

The `$/100k` column is the one that changes minds. A difference of $0.0002 per
call is invisible until you multiply by production volume, at which point it is
the difference between a $20 and a $500 monthly bill for the same feature.

The `reason tok` column is the reasoning-model tax made visible. Those tokens are
billed as completion tokens and never appear in
`choices[0].message.content`. If `o4-mini` shows hundreds or thousands of
reasoning tokens for an answer that is a single integer, you now know exactly why
its cost per call is not what the headline rate implied.

Watch `max s` as well as `p50 s`. A p50 of 1.2 s with a max of 9 s is a model you
cannot put in front of a user without streaming.

## 6. When *not* to use a model at all

The most-missed exam decision is "Foundry Tool, not LLM." Below, one task —
detect the language of a string — is done both ways so the difference is concrete.

The Language service shares the Foundry (AI Services) endpoint and authenticates
with the same Entra ID credential, so there is no extra resource to create.

In [ ]:
SAMPLES = [
    "El servicio no responde desde ayer por la tarde.",
    "Die Rechnung stimmt nicht mit dem Angebot überein.",
    "Le tableau de bord affiche une erreur 500.",
]

# --- Way 1: prompt an LLM ---
t0 = time.perf_counter()
llm_resp = client.chat.completions.create(
    model=cfg["MODEL_MINI"],
    messages=[
        {"role": "system", "content": "Return only ISO 639-1 codes, one per line, in order."},
        {"role": "user", "content": "\n".join(SAMPLES)},
    ],
    temperature=0.0,
)
llm_secs = time.perf_counter() - t0
print("LLM      :", llm_resp.choices[0].message.content.split())
print(f"           {llm_secs:.2f}s, {llm_resp.usage.total_tokens} tokens, no confidence score")

# --- Way 2: the Foundry Tool ---
try:
    from azure.ai.textanalytics import TextAnalyticsClient

    lang_endpoint = cfg.get("AZURE_LANGUAGE_ENDPOINT") or cfg["AZURE_OPENAI_ENDPOINT"]
    ta = TextAnalyticsClient(endpoint=lang_endpoint, credential=credential())

    t0 = time.perf_counter()
    tool_result = ta.detect_language(SAMPLES)
    tool_secs = time.perf_counter() - t0
    print(
        "\nLanguage :",
        [f"{d.primary_language.iso6391_name}({d.primary_language.confidence_score:.2f})" for d in tool_result],
    )
    print(f"           {tool_secs:.2f}s, billed per 1,000 text records, deterministic")
except Exception as exc:  # noqa: BLE001
    print(f"\nLanguage : unavailable — {type(exc).__name__}: {exc}")
    print("           Needs the Language capability on the Foundry resource and")
    print("           the 'Cognitive Services Language Reader' or 'Cognitive Services User' role.")

Three differences the exam cares about, none of which is "the LLM got it wrong":

1. The Tool returns a **confidence score**. The LLM returns text with no
   calibrated uncertainty.
2. The Tool is **deterministic**. Re-run the LLM at `temperature=0` and you will
   *usually* get the same answer, which is not the same as always.
3. The Tool bills per **text record**, not per token, so cost is predictable
   regardless of input length.

Same reasoning applies to PII redaction (Language), OCR and field extraction
(Document Intelligence / Content Understanding), transcription (Speech), and
translation (Translator).

## 7. Retrieval choice, demonstrated without an index

You do not need Azure AI Search running to feel why hybrid beats pure vector.
Below, a tiny corpus is scored two ways: cosine similarity over embeddings
(vector), and a crude term overlap (a stand-in for BM25). Watch what happens to
the query containing a part number.

In [ ]:
from ai103 import embed

CORPUS = [
    "Reset the device by holding the power button for ten seconds.",
    "Part X-2200 is the replacement filter cartridge for the XR series.",
    "Refunds are issued to the original payment method within 14 days.",
    "If the unit will not switch on, check the mains fuse and the cable.",
]

corpus_vecs = embed(CORPUS)


def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / ((sum(x * x for x in a) ** 0.5) * (sum(y * y for y in b) ** 0.5))


def lexical(query, doc):
    q = set(re.findall(r"[a-z0-9\-]+", query.lower()))
    d = set(re.findall(r"[a-z0-9\-]+", doc.lower()))
    return len(q & d) / len(q) if q else 0.0


for query in ["my machine is dead, nothing happens", "X-2200"]:
    qv = embed(query)[0]
    print(f"\nquery: {query!r}")
    print(f"  {'vector':>8} {'lexical':>8}   document")
    for doc, vec in zip(CORPUS, corpus_vecs):
        print(f"  {cosine(qv, vec):>8.3f} {lexical(query, doc):>8.3f}   {doc[:52]}")

For the paraphrased query, vector wins outright — no shared words, correct
document. For `X-2200`, the embedding barely separates the documents because a
rare alphanumeric token carries almost no semantic signal, while lexical matching
nails it instantly.

Neither mode is safe alone. **Hybrid** runs both and fuses the rankings with
Reciprocal Rank Fusion; the **semantic ranker** then reorders the fused top-N with
a language model. That stack is the default correct answer for a grounding
scenario, and unit 05.1 builds it for real.

## 8. Cleanup

This lab created no agents, indexes, or deployments — only model calls, which
bill per token and stop costing money the moment they return. Nothing to delete.

The cell below just confirms that, and is worth running as a habit: from unit 01.2
onward the labs *do* create billable things.

In [ ]:
before = set(deployed)
after = {d.name for d in project.deployments.list()}
agents = list(project.agents.list_agents()) if hasattr(project, "agents") else []

print("deployments created by this lab:", after - before or "none")
print("agents in project             :", len(agents))
print("hourly-billed resources created: none")

## Exercise

Solutions at the bottom of [quiz.md](quiz.md).

1. **Justify a choice.** Using your own table from section 5, write three or four
   sentences choosing a model for this requirement: *a customer-facing chat widget,
   80,000 calls per month, p95 latency budget 3 seconds, wrong answers are
   embarrassing but not dangerous.* Quote your measured numbers. Then change the
   requirement to *an internal nightly job that reconciles 5,000 invoices, where a
   wrong answer costs real money and latency does not matter* and say whether your
   answer changes and why.

2. **Make the small model win.** Add a worked example and an explicit
   step-by-step instruction to the prompt, then re-run only `gpt-4o-mini`. Does
   accuracy reach the frontier model's? Compare the *new* cost per call —
   the longer prompt is not free. Is prompt engineering or a bigger model cheaper
   here at 80,000 calls a month?

3. **Price the reasoning tax.** For `o4-mini`, print
   `usage.completion_tokens_details.reasoning_tokens` alongside
   `usage.completion_tokens` for a single call. What fraction of what you paid for
   did you never see? Repeat with a trivial prompt ("What is 2+2?") and compare
   the ratio.

4. **Read the catalog for a constraint.** Using the `catalog` list from section 2,
   print every model in your region whose SKU list contains `GlobalBatch`. Which
   of the three deployment-type scenarios in the README would those satisfy?

In [ ]:
# Your work here.

## Next

[01.2 — Set up AI solutions in Foundry](../02_setup_foundry_solutions/README.md)